In [19]:
import spacy

# 영어 모델 로드 (가벼운 sm 모델 또는 정밀한 trf 모델 선택 가능)
nlp = spacy.load("D:\\models\\en_core_web_sm", disable=["ner", "parser"])
nlp

In [25]:
from rank_bm25 import BM25Okapi
from tqdm import tqdm
# 2. 샘플 데이터
corpus = [
    "The fatty acids in fish are good for heart health.",
    "Data scientists use Python to build machine learning models.",
    "The quick brown fox jumps over the lazy dog.",
    "BM25 is a ranking function used by search engines to estimate the relevance of documents.",
    "Natural language processing involves the interaction between computers and humans."
]

# 3. spaCy 전처리 함수
def spacy_tokenizer(text):
    # 문서 객체 생성
    doc = nlp(text)
    # 1. 불용어(Stopwords) 제거
    # 2. 구두점(Punctuation) 제거 
    # 3. 표제어 추출(Lemmatization) 및 소문자화
    return [token.lemma_.lower() for token in doc 
            if not token.is_stop and not token.is_punct and not token.is_space]

# 4. 전처리: 대량 문서 토큰화 - 대량 문서 처리 시 권장 방식
# n_process=-1은 모든 CPU 코어를 사용하겠다는 의미입니다. (주피터노트북에서는 에러 발생 그냥 1로 테스트)
print("Tokenizing with spaCy...")
tokenized_corpus = []
for doc in tqdm(nlp.pipe(corpus, batch_size=2000, n_process=1), total=len(corpus)):
    tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct]
    tokenized_corpus.append(tokens)

# 5. BM25 인덱싱
bm25 = BM25Okapi(tokenized_corpus)

# 6. 검색 수행
query = "Are fish oils healthy for the heart?"
tokenized_query = spacy_tokenizer(query)

# 결과 상위 2개 추출
top_n = bm25.get_top_n(tokenized_query, corpus, n=3)

print(f"\nQuery: {query}")
print(f"Top Results:")
for i, res in enumerate(top_n):
    print(f"{i+1}: {res}")

Tokenizing with spaCy...


100%|██████████| 5/5 [00:00<00:00, 455.97it/s]


Query: Are fish oils healthy for the heart?
Top Results:
1: The fatty acids in fish are good for heart health.
2: Natural language processing involves the interaction between computers and humans.
3: BM25 is a ranking function used by search engines to estimate the relevance of documents.
